In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib as mpl

import ipywidgets as widgets

from matplotlib.lines import Line2D
import numpy as np
import scipy as sp
np.random.seed(42)

In [ ]:
import UBNA_localize_process_GCC__20260629_for_ameena as localize

In [ ]:
fig = plt.figure(figsize=(4, 4))
ax = fig.add_subplot(projection="3d")

origin = np.array([0, 0, 0])
axis_len = 1

ax.quiver(*origin, axis_len, 0, 0, color="r", arrow_length_ratio=0.12)
ax.quiver(*origin, 0, axis_len, 0, color="g", arrow_length_ratio=0.12)
ax.quiver(*origin, 0, 0, axis_len, color="b", arrow_length_ratio=0.12)

ax.text(axis_len * 1.1, 0, 0, "+x horizontal right", color="r")
ax.text(0, axis_len * 1.1, 0, "+y vertical right", color="g")
ax.text(0, 0, axis_len * 1.1, "+z up", color="b")

ax.set_xlim(0, 2)
ax.set_ylim(0, 2)
ax.set_zlim(0, 2)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_box_aspect([1, 1, 1])
ax.view_init(elev=20, azim=-60)

plt.show()

In [ ]:
SAMPLERATE = 250000
NUM_CHANNELS = 8
BYTES_PER_SAMPLE = 2 #int16

COLOR_MAP_FOR_TRAJ = {0:cm.Blues, 1:cm.Reds, 2:cm.Greens}
MIC_MARKER_THICKNESS = 2
POINT_SIZE = 75

In [ ]:
%matplotlib inline

SELECTED_CHANNEL_FOR_REF = 0
MICROPHONES_USED = np.array([1,2,3,4,5,6,7,8])
IND_OF_SELECTED_CHANNEL = np.where(MICROPHONES_USED==(SELECTED_CHANNEL_FOR_REF+1))[0]
assert(IND_OF_SELECTED_CHANNEL==SELECTED_CHANNEL_FOR_REF)
SELECTED_MIC_FOR_REF = MICROPHONES_USED[IND_OF_SELECTED_CHANNEL]
NON_REF_MICROPHONES_USED = MICROPHONES_USED[MICROPHONES_USED!=SELECTED_MIC_FOR_REF]
NUM_GOOD_CHANNELS = MICROPHONES_USED.shape[0]
NUM_NONREFCHANNELS = NUM_GOOD_CHANNELS-1

GRID_SIZE = 50
CIRCLE_RADIUS = 10
SOUND_SPEED_AIR = 343
TIME_DURATION = 0.2
RCVR_COLORS = ['red', 'limegreen', 'blue', 'mediumpurple', 'slategray', 'brown', 'green', 'darkred']
mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=RCVR_COLORS)
COLOR_CYCLE = plt.rcParams['axes.prop_cycle'].by_key()['color']
BAT_INIT_DIST = 10
TEMPLATE_CHANNEL = 3

FS = 250000
ASSUMED_BAT_SPEED = 4*np.sqrt(3)/3 # m/s
ASSUMED_BAT_IPI = 0.1 # secs (100ms)
TIMESTEPS = np.arange(0, ((2*BAT_INIT_DIST)/ASSUMED_BAT_SPEED) + ASSUMED_BAT_IPI, ASSUMED_BAT_IPI)

In [ ]:
def make_sphere_surface_points(num_points=100, radius=20):
    i = np.arange(num_points)
    golden_angle = np.pi * (3 - np.sqrt(5))

    z = 1 - 2 * (i + 0.5) / num_points
    xy = np.sqrt(1 - z**2)
    theta = golden_angle * i

    x = radius * xy * np.cos(theta)
    y = radius * xy * np.sin(theta)
    z = radius * z

    return np.column_stack([x, y, z])

In [ ]:
A_locs_mat_tdoa = np.array([[0, 0, 0],  #M1 (top left front)
                            [0, 0, -1], #M2 (bottom left front)
                            [1, 0, 0],  #M3 (top right front)
                            [1, 0, -1], #M4 (bottom right front)
                            [0, 1, 0],  #M5 (top left back)
                            [0, 1, -1], #M6 (bottom left back)
                            [1, 1, 0],  #M7 (top right back)
                            [1, 1, -1]])#M8 (bottom right back)

### Simulation #1: near-field nonlinear least-squares (NLS) localization

The signal simulation and TDOA estimator are unchanged. Only the mapping
from measured range differences to a 3-D source position is replaced.


In [ ]:
%matplotlib widget

sim_mic_labels = ["M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8"]
sim_all_mic_edges = [(edge_start, edge_end) for edge_start in range(len(sim_mic_labels)) for edge_end in range(edge_start + 1, len(sim_mic_labels))]
sim_starting_mic_locs = np.array(A_locs_mat_tdoa, dtype=float).copy()
sim_mic_coord_inputs = []

for mic_label, mic_loc in zip(sim_mic_labels, sim_starting_mic_locs):
    mic_row = [widgets.Label(value=mic_label, layout=widgets.Layout(width="40px"))]

    for coord_label, coord_value in zip(["x", "y", "z"], mic_loc):
        mic_row.append(
            widgets.FloatText(
                value=float(coord_value),
                description=coord_label,
                layout=widgets.Layout(width="140px"),
                style={"description_width": "20px"},
            )
        )

    sim_mic_coord_inputs.append(mic_row)

sim_mic_coord_rows = [widgets.HBox(mic_row) for mic_row in sim_mic_coord_inputs]
sim_mic_coord_grid = widgets.VBox(sim_mic_coord_rows)
sim_output = widgets.Output()


def get_sim_widget_mic_locs():
    return np.array([[coord_input.value for coord_input in mic_row[1:]] for mic_row in sim_mic_coord_inputs])


def simulate_N_receivers_with_mic_placement(ascale, sphere_radius, num_pts, noise_power_dB):
    global A_locs_mat_tdoa, ref_A_loc

    A_locs_mat_tdoa = get_sim_widget_mic_locs()
    ref_A_loc = A_locs_mat_tdoa[0]

    resized_UBNA_ARRAY_MIC_LOCS = ascale * A_locs_mat_tdoa
    N = resized_UBNA_ARRAY_MIC_LOCS.shape[0]
    rN_x = resized_UBNA_ARRAY_MIC_LOCS[:,0]
    rN_y = resized_UBNA_ARRAY_MIC_LOCS[:,1]
    rN_z = resized_UBNA_ARRAY_MIC_LOCS[:,2]
    A_LOCS_MAT = resized_UBNA_ARRAY_MIC_LOCS
    A_locs_mat_wrt_ref_channel = A_LOCS_MAT
    A_locs_mat_tdoa_meters = A_locs_mat_wrt_ref_channel[(NON_REF_MICROPHONES_USED-1)]

    x0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][0]
    y0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][1]
    z0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][2]
    xm = A_locs_mat_tdoa_meters[:,0].reshape((NUM_NONREFCHANNELS, 1))
    ym = A_locs_mat_tdoa_meters[:,1].reshape((NUM_NONREFCHANNELS, 1))
    zm = A_locs_mat_tdoa_meters[:,2].reshape((NUM_NONREFCHANNELS, 1))

    time_transmitted = TIME_DURATION/2
    t = np.linspace(0, TIME_DURATION, int(FS*TIME_DURATION))
    raw_signal = sp.signal.chirp(t, f0=40000, t1=0.01, f1=25000)
    template_signal = raw_signal[:int(FS*0.01)]
    signal = np.zeros(t.size)
    signal[int(FS*time_transmitted):int(FS*(time_transmitted+0.01))] = template_signal
    signal_duration = TIME_DURATION/2

    sphere_surface_points = make_sphere_surface_points(num_points=num_pts, radius=sphere_radius)
    x_arr = sphere_surface_points[:,0]
    y_arr = sphere_surface_points[:,1]
    z_arr = sphere_surface_points[:,2]

    dist_from_source_to_r_n_arr = np.sqrt((x_arr[None, :] - rN_x[:, None]) ** 2 + (y_arr[None, :] - rN_y[:, None]) ** 2 + (z_arr[None, :] - rN_z[:, None]) ** 2)
    time_to_rn_array = dist_from_source_to_r_n_arr / SOUND_SPEED_AIR
    receive_signaln_for_timestep = signal[:, None, None]
    signal_duration_samples = np.arange(int(signal_duration * FS))
    rn_indices = ((time_transmitted - time_to_rn_array) * FS).astype(int) + signal_duration_samples[:, None, None]
    chunk_of_received_signaln_in_window_timestep = np.take_along_axis(receive_signaln_for_timestep, rn_indices, axis=0)

    example_calls = chunk_of_received_signaln_in_window_timestep[:,:,:].T
    noise_power_lin = 10**(noise_power_dB/10)
    noise_mat = np.random.normal(loc=0, scale=noise_power_lin, size=example_calls.shape)
    noisy_calls = example_calls + noise_mat

    example_t_delays_wrt_selected_channel = np.zeros((noisy_calls.shape[0], NUM_GOOD_CHANNELS), dtype=np.float64)
    for non_ref_call_i in range(NUM_GOOD_CHANNELS):
        for det_num in range(noisy_calls.shape[0]):
            nonref_dets = noisy_calls[det_num]
            approx_call_dur = 0.01
            ref_call_of_det, ref_channel_num = localize.index_reference_call_based_on_channel(noisy_calls, det_num, TEMPLATE_CHANNEL)
            ref_mic_call_only, found_call_start, found_call_dur = localize.extract_reference_call_only(ref_call_of_det, approx_call_dur)

            signal1 = nonref_dets[non_ref_call_i, :]
            signal2 = ref_mic_call_only
            cross_correlation = sp.signal.correlate((signal1), (signal2), mode='full')
            lags = sp.signal.correlation_lags(len(signal1), len(signal2), mode='full')
            max_lag_index = np.argmax(cross_correlation)
            time_delay_in_samples = lags[max_lag_index]
            time_delay_in_seconds = time_delay_in_samples / FS
            example_t_delays_wrt_selected_channel[det_num, non_ref_call_i] = time_delay_in_seconds

    time_received_at_rn = example_t_delays_wrt_selected_channel
    t_delay_ref_to_selected_channel = time_received_at_rn[:,IND_OF_SELECTED_CHANNEL]
    time_delay_mics_to_selected_ref_channel = (time_received_at_rn[:]) - (t_delay_ref_to_selected_channel.reshape((len(t_delay_ref_to_selected_channel), 1)))
    time_delay_mics_to_selected_ref_channel_no_ref = np.delete(time_delay_mics_to_selected_ref_channel, IND_OF_SELECTED_CHANNEL, axis=1)
    measured_d_mics_to_ref = (time_delay_mics_to_selected_ref_channel_no_ref * SOUND_SPEED_AIR)

    fig = plt.figure(figsize=(15, 10))
    plt.rcParams.update({'font.size':12})
    marker_handles = [Line2D([0], [0], marker='o', color='w', label=f'rcvr #{i+1}', markerfacecolor=COLOR_CYCLE[i], markersize=6) for i in range(N)]
    map_ax_3d = fig.add_subplot(projection='3d')

    for edge_start, edge_end in sim_all_mic_edges:
        edge_locs = resized_UBNA_ARRAY_MIC_LOCS[[edge_start, edge_end]]
        map_ax_3d.plot(edge_locs[:,0], edge_locs[:,1], edge_locs[:,2], color="black", linewidth=0.8, alpha=0.25)

    map_ax_3d.scatter(rN_x, rN_y, rN_z, s=30, c=COLOR_CYCLE[:N], edgecolor='k')

    for mic_label, mic_loc in zip(sim_mic_labels, resized_UBNA_ARRAY_MIC_LOCS):
        map_ax_3d.text(mic_loc[0], mic_loc[1], mic_loc[2], mic_label, fontsize=8)

    if measured_d_mics_to_ref.shape[0] >= 1:
        source_locs = np.zeros((3, measured_d_mics_to_ref.shape[0]), dtype=np.float64)
        reference_mic = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF]
        nonreference_mics = A_locs_mat_tdoa_meters
        baselines = nonreference_mics - reference_mic

        for i in range(measured_d_mics_to_ref.shape[0]):
            observed_range_differences = measured_d_mics_to_ref[i, :]

            # Use the linear GS estimate only as a deterministic initial point.
            measured_dm0_column = observed_range_differences.reshape((NUM_NONREFCHANNELS, 1))
            A_mat = np.hstack((baselines, measured_dm0_column))
            wm0 = (
                np.sum(nonreference_mics**2, axis=1, keepdims=True)
                - np.sum(reference_mic**2)
                - measured_dm0_column**2
            ) / 2
            initial_augmented, *_ = sp.linalg.lstsq(A_mat, wm0)
            initial_position = initial_augmented[:3, 0]

            def range_difference_residuals(position):
                predicted = (
                    np.linalg.norm(position - nonreference_mics, axis=1)
                    - np.linalg.norm(position - reference_mic)
                )
                return predicted - observed_range_differences

            fit = sp.optimize.least_squares(
                range_difference_residuals,
                initial_position,
                method="trf",
                loss="linear",
                max_nfev=500,
            )
            source_locs[:, i] = fit.x

    map_ax_3d.scatter(source_locs[0,:], source_locs[1,:], source_locs[2,:], color='red', edgecolor='k', alpha=1, s=POINT_SIZE/2, zorder=2)
    map_ax_3d.scatter(x_arr, y_arr, z_arr, color='cyan', edgecolor='k', alpha=1, s=POINT_SIZE/2, zorder=2)
    avg_d_err = (source_locs[0,:]-x_arr)**2 + (source_locs[1,:]-y_arr)**2 + (source_locs[2,:]-z_arr)**2
    avg_d_err = np.sqrt(np.mean(avg_d_err))
    map_ax_3d.text(x=10, y=10, z=25, s=f"Avg. Distance error = {avg_d_err:.6f}", color='k', fontsize=14)
    for i in range(source_locs.shape[1]):
        map_ax_3d.plot([source_locs[0,i], x_arr[i]], [source_locs[1,i], y_arr[i]], [source_locs[2,i], z_arr[i]], color="black", linewidth=0.8, alpha=0.4)
    map_ax_3d.set_xlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_3d.set_ylim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_3d.set_zlim(-GRID_SIZE / 2, GRID_SIZE /2)
    map_ax_3d.set_xlabel("X (meters)")
    map_ax_3d.set_ylabel("Y (meters)")
    map_ax_3d.set_zlabel("Z (meters)")
    map_ax_3d.set_box_aspect([1, 1, 1])
    map_ax_3d.set_title(f"Near-field NLS Localization ({int(N)} receivers)")
    map_ax_3d.legend(handles=marker_handles, ncol=2, loc='upper right')

    plt.show()


def update_sim_with_mic_placement(change=None):
    with sim_output:
        sim_output.clear_output(wait=True)
        simulate_N_receivers_with_mic_placement(
            ascale=placement_as_scale_slider.value,
            sphere_radius=placement_radius_slider.value,
            num_pts=placement_num_surface_points_slider.value,
            noise_power_dB=placement_noise_slider.value,
        )


placement_as_scale_slider = widgets.FloatSlider(
    value=1, min=0.5, max=5, step=0.1, description="Scale factor (α)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="1000px")
)

placement_radius_slider = widgets.FloatSlider(
    value=10, min=1, max=25, step=1, description="Sphere radius (r)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="1000px")
)

placement_num_surface_points_slider = widgets.IntSlider(
    value=10, min=10, max=1000, step=10, description="Number of Surface Points", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="800px")
)

placement_noise_slider = widgets.IntSlider(
    value=-10, min=-90, max=0, step=1, description="Noise Power (dBFS)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="800px")
)

placement_control_widgets = [
    placement_as_scale_slider,
    placement_radius_slider,
    placement_num_surface_points_slider,
    placement_noise_slider,
]

for control_widget in placement_control_widgets:
    control_widget.observe(update_sim_with_mic_placement, names="value")

for mic_row in sim_mic_coord_inputs:
    for coord_input in mic_row[1:]:
        coord_input.observe(update_sim_with_mic_placement, names="value")

update_sim_with_mic_placement()
display(widgets.VBox([widgets.VBox(placement_control_widgets), sim_mic_coord_grid, sim_output]))


### Simulation #2: DOA-based method for downchirp transmission using GCC for cube array design
This version keeps the signal simulation, propagation, noise, and GCC/TDOA estimation unchanged.

In [ ]:
%matplotlib widget

doa_sim_mic_labels = ["M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8"]
doa_sim_all_mic_edges = [(edge_start, edge_end) for edge_start in range(len(doa_sim_mic_labels)) for edge_end in range(edge_start + 1, len(doa_sim_mic_labels))]
doa_sim_starting_mic_locs = np.array(A_locs_mat_tdoa, dtype=float).copy()
doa_sim_mic_coord_inputs = []

for mic_label, mic_loc in zip(doa_sim_mic_labels, doa_sim_starting_mic_locs):
    mic_row = [widgets.Label(value=mic_label, layout=widgets.Layout(width="40px"))]

    for coord_label, coord_value in zip(["x", "y", "z"], mic_loc):
        mic_row.append(
            widgets.FloatText(
                value=float(coord_value),
                description=coord_label,
                layout=widgets.Layout(width="140px"),
                style={"description_width": "20px"},
            )
        )

    doa_sim_mic_coord_inputs.append(mic_row)

doa_sim_mic_coord_rows = [widgets.HBox(mic_row) for mic_row in doa_sim_mic_coord_inputs]
doa_sim_mic_coord_grid = widgets.VBox(doa_sim_mic_coord_rows)
doa_sim_output = widgets.Output()


def get_doa_sim_widget_mic_locs():
    return np.array([[coord_input.value for coord_input in mic_row[1:]] for mic_row in doa_sim_mic_coord_inputs])


def simulate_N_receivers_with_mic_placement_doa(ascale, sphere_radius, num_pts, noise_power_dB):
    global A_locs_mat_tdoa, ref_A_loc

    A_locs_mat_tdoa = get_doa_sim_widget_mic_locs()
    ref_A_loc = A_locs_mat_tdoa[0]

    resized_UBNA_ARRAY_MIC_LOCS = ascale * A_locs_mat_tdoa
    N = resized_UBNA_ARRAY_MIC_LOCS.shape[0]
    rN_x = resized_UBNA_ARRAY_MIC_LOCS[:,0]
    rN_y = resized_UBNA_ARRAY_MIC_LOCS[:,1]
    rN_z = resized_UBNA_ARRAY_MIC_LOCS[:,2]
    A_LOCS_MAT = resized_UBNA_ARRAY_MIC_LOCS
    A_locs_mat_wrt_ref_channel = A_LOCS_MAT
    A_locs_mat_tdoa_meters = A_locs_mat_wrt_ref_channel[(NON_REF_MICROPHONES_USED-1)]

    # x0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][0]
    # y0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][1]
    # z0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][2]
    # xm = A_locs_mat_tdoa_meters[:,0].reshape((NUM_NONREFCHANNELS, 1))
    # ym = A_locs_mat_tdoa_meters[:,1].reshape((NUM_NONREFCHANNELS, 1))
    # zm = A_locs_mat_tdoa_meters[:,2].reshape((NUM_NONREFCHANNELS, 1))

    time_transmitted = TIME_DURATION/2
    t = np.linspace(0, TIME_DURATION, int(FS*TIME_DURATION))
    raw_signal = sp.signal.chirp(t, f0=40000, t1=0.01, f1=25000)
    template_signal = raw_signal[:int(FS*0.01)]
    signal = np.zeros(t.size)
    signal[int(FS*time_transmitted):int(FS*(time_transmitted+0.01))] = template_signal
    signal_duration = TIME_DURATION/2

    sphere_surface_points = make_sphere_surface_points(num_points=num_pts, radius=sphere_radius)
    x_arr = sphere_surface_points[:,0]
    y_arr = sphere_surface_points[:,1]
    z_arr = sphere_surface_points[:,2]

    dist_from_source_to_r_n_arr = np.sqrt((x_arr[None, :] - rN_x[:, None]) ** 2 + (y_arr[None, :] - rN_y[:, None]) ** 2 + (z_arr[None, :] - rN_z[:, None]) ** 2)
    time_to_rn_array = dist_from_source_to_r_n_arr / SOUND_SPEED_AIR
    receive_signaln_for_timestep = signal[:, None, None]
    signal_duration_samples = np.arange(int(signal_duration * FS))
    rn_indices = ((time_transmitted - time_to_rn_array) * FS).astype(int) + signal_duration_samples[:, None, None]
    chunk_of_received_signaln_in_window_timestep = np.take_along_axis(receive_signaln_for_timestep, rn_indices, axis=0)

    example_calls = chunk_of_received_signaln_in_window_timestep[:,:,:].T
    noise_power_lin = 10**(noise_power_dB/10)
    noise_mat = np.random.normal(loc=0, scale=noise_power_lin, size=example_calls.shape)
    noisy_calls = example_calls + noise_mat

    example_t_delays_wrt_selected_channel = np.zeros((noisy_calls.shape[0], NUM_GOOD_CHANNELS), dtype=np.float64)
    for non_ref_call_i in range(NUM_GOOD_CHANNELS):
        for det_num in range(noisy_calls.shape[0]):
            nonref_dets = noisy_calls[det_num]
            approx_call_dur = 0.01
            ref_call_of_det, ref_channel_num = localize.index_reference_call_based_on_channel(noisy_calls, det_num, TEMPLATE_CHANNEL)
            ref_mic_call_only, found_call_start, found_call_dur = localize.extract_reference_call_only(ref_call_of_det, approx_call_dur)

            signal1 = nonref_dets[non_ref_call_i, :]
            signal2 = ref_mic_call_only
            cross_correlation = sp.signal.correlate((signal1), (signal2), mode='full')
            lags = sp.signal.correlation_lags(len(signal1), len(signal2), mode='full')
            max_lag_index = np.argmax(cross_correlation)
            time_delay_in_samples = lags[max_lag_index]
            time_delay_in_seconds = time_delay_in_samples / FS
            example_t_delays_wrt_selected_channel[det_num, non_ref_call_i] = time_delay_in_seconds

    time_received_at_rn = example_t_delays_wrt_selected_channel
    t_delay_ref_to_selected_channel = time_received_at_rn[:,IND_OF_SELECTED_CHANNEL]
    time_delay_mics_to_selected_ref_channel = (time_received_at_rn[:]) - (t_delay_ref_to_selected_channel.reshape((len(t_delay_ref_to_selected_channel), 1)))
    time_delay_mics_to_selected_ref_channel_no_ref = np.delete(time_delay_mics_to_selected_ref_channel, IND_OF_SELECTED_CHANNEL, axis=1)
    measured_d_mics_to_ref = (time_delay_mics_to_selected_ref_channel_no_ref * SOUND_SPEED_AIR)

    fig = plt.figure(figsize=(15, 10))
    plt.rcParams.update({'font.size':12})
    marker_handles = [Line2D([0], [0], marker='o', color='w', label=f'rcvr #{i+1}', markerfacecolor=COLOR_CYCLE[i], markersize=6) for i in range(N)]
    map_ax_3d = fig.add_subplot(projection='3d')

    for edge_start, edge_end in doa_sim_all_mic_edges:
        edge_locs = resized_UBNA_ARRAY_MIC_LOCS[[edge_start, edge_end]]
        map_ax_3d.plot(edge_locs[:,0], edge_locs[:,1], edge_locs[:,2], color="black", linewidth=0.8, alpha=0.25)

    map_ax_3d.scatter(rN_x, rN_y, rN_z, s=30, c=COLOR_CYCLE[:N], edgecolor='k')

    for mic_label, mic_loc in zip(doa_sim_mic_labels, resized_UBNA_ARRAY_MIC_LOCS):
        map_ax_3d.text(mic_loc[0], mic_loc[1], mic_loc[2], mic_label, fontsize=8)

    # Use the same measured path differences with the far-field plane-wave DOA model.
    microphone_baselines = A_locs_mat_tdoa_meters - A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF]
    estimated_directions = np.zeros((3, measured_d_mics_to_ref.shape[0]), dtype=np.float64)

    for i in range(measured_d_mics_to_ref.shape[0]):
        measured_dm0_column = measured_d_mics_to_ref[i,:].reshape((NUM_NONREFCHANNELS, 1))
        doa_xs, residuals, rank, sing_vals = sp.linalg.lstsq(microphone_baselines, -measured_dm0_column)
        estimated_directions[:,i] = doa_xs[:,0]

    # Keep the true simulated source locations visible in cyan.
    map_ax_3d.scatter(x_arr, y_arr, z_arr, color='cyan', edgecolor='k', alpha=1, s=POINT_SIZE/2, zorder=2)

    # Calculate the true unit direction from the reference microphone to every simulated source.
    reference_mic_loc = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF]
    true_source_locs_wrt_ref = sphere_surface_points - reference_mic_loc
    true_source_distances = np.linalg.norm(true_source_locs_wrt_ref, axis=1)
    true_directions = true_source_locs_wrt_ref / true_source_distances[:,None]

    # Compare estimated and true unit vectors using the same sum-of-squared-coordinate-error structure.
    direction_sq_err = np.sum((estimated_directions.T - true_directions)**2, axis=1)
    mean_direction_sq_err = np.mean(direction_sq_err)
    map_ax_3d.text(x=10, y=10, z=25, s=f"Mean Squared Direction Error = {mean_direction_sq_err:.6f}", color='k', fontsize=14)

    # Use each true source distance only to choose a visible ray length; DOA itself does not estimate range.
    estimated_ray_ends = reference_mic_loc[:,None] + estimated_directions * true_source_distances[None,:]
    map_ax_3d.scatter(estimated_ray_ends[0,:], estimated_ray_ends[1,:], estimated_ray_ends[2,:], color='red', edgecolor='k', alpha=1, s=POINT_SIZE/2, zorder=2)

    # Draw one red ray from the reference microphone for every reconstructed direction of arrival.
    for i in range(estimated_directions.shape[1]):
        map_ax_3d.plot([reference_mic_loc[0], estimated_ray_ends[0,i]], [reference_mic_loc[1], estimated_ray_ends[1,i]], [reference_mic_loc[2], estimated_ray_ends[2,i]], color="red", linewidth=0.8, alpha=0.4)
    map_ax_3d.set_xlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_3d.set_ylim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_3d.set_zlim(-GRID_SIZE / 2, GRID_SIZE /2)
    map_ax_3d.set_xlabel("X (meters)")
    map_ax_3d.set_ylabel("Y (meters)")
    map_ax_3d.set_zlabel("Z (meters)")
    map_ax_3d.set_box_aspect([1, 1, 1])
    # map_ax_3d.set_title(f"Sound Reception ({int(N)} receivers; 1 source)")
    # Update only the title to identify the DOA-based reconstruction.
    map_ax_3d.set_title(f"Direction of Arrival Reconstruction ({int(N)} receivers; 1 source)")
    map_ax_3d.legend(handles=marker_handles, ncol=2, loc='upper right')

    plt.show()


def update_doa_sim_with_mic_placement(change=None):
    with doa_sim_output:
        doa_sim_output.clear_output(wait=True)
        simulate_N_receivers_with_mic_placement_doa(
            ascale=doa_placement_as_scale_slider.value,
            sphere_radius=doa_placement_radius_slider.value,
            num_pts=doa_placement_num_surface_points_slider.value,
            noise_power_dB=doa_placement_noise_slider.value,
        )


doa_placement_as_scale_slider = widgets.FloatSlider(
    value=1, min=0.5, max=5, step=0.1, description="Scale factor (α)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="1000px")
)

doa_placement_radius_slider = widgets.FloatSlider(
    value=10, min=1, max=25, step=1, description="Sphere radius (r)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="1000px")
)

doa_placement_num_surface_points_slider = widgets.IntSlider(
    value=10, min=10, max=1000, step=10, description="Number of Surface Points", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="800px")
)

doa_placement_noise_slider = widgets.IntSlider(
    value=-10, min=-90, max=0, step=1, description="Noise Power (dBFS)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="800px")
)

doa_placement_control_widgets = [
    doa_placement_as_scale_slider,
    doa_placement_radius_slider,
    doa_placement_num_surface_points_slider,
    doa_placement_noise_slider,
]

for control_widget in doa_placement_control_widgets:
    control_widget.observe(update_doa_sim_with_mic_placement, names="value")

for mic_row in doa_sim_mic_coord_inputs:
    for coord_input in mic_row[1:]:
        coord_input.observe(update_doa_sim_with_mic_placement, names="value")

update_doa_sim_with_mic_placement()
display(widgets.VBox([widgets.VBox(doa_placement_control_widgets), doa_sim_mic_coord_grid, doa_sim_output]))